In [3]:
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 50.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 53.9 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-learn] [scikit-learn]


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [5]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)

In [6]:
categorical = list(df.select_dtypes(include=['object']).columns)
numerical = list(df.select_dtypes(include=['number']).columns)

if 'converted' in numerical:
    numerical.remove('converted')


/tmp/ipykernel_2282/3431309696.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical = list(df.select_dtypes(include=['object']).columns)


In [7]:
for c in categorical:
    df[c] = df[c].fillna('NA')

for n in numerical:
    df[n] = df[n].fillna(0.0)

In [8]:
q1_mode = df['industry'].mode()[0]
print(f"--- Question 1 ---")
print(f"Most frequent value (mode) for industry: {q1_mode}\n")

--- Question 1 ---
Most frequent value (mode) for industry: technology



In [9]:
print(f"--- Question 2 ---")
corr = df[numerical].corr()
print(corr)
print()

--- Question 2 ---
                          annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.161300   
number_of_courses_viewed       0.161300                  1.000000   
interaction_count              0.122842                  0.721609   
lead_score                     0.229496                  0.757204   

                          interaction_count  lead_score  
annual_income                      0.122842    0.229496  
number_of_courses_viewed           0.721609    0.757204  
interaction_count                  1.000000    0.915746  
lead_score                         0.915746    1.000000  



In [10]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']


In [11]:
def calculate_mi(series):
    return mutual_info_score(series, y_train)

mi_scores = df_train[categorical].apply(calculate_mi)
mi_scores_rounded = mi_scores.round(2).sort_values(ascending=False)

print(f"--- Question 3 ---")
print("Mutual Information Scores:")
print(mi_scores_rounded)
print()


--- Question 3 ---
Mutual Information Scores:
lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64



In [12]:
all_features = categorical + numerical

train_dicts = df_train[all_features].to_dict(orient='records')
val_dicts = df_val[all_features].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
original_acc = (y_pred == y_val).mean()

print(f"--- Question 4 ---")
print(f"Validation Accuracy: {round(original_acc, 2)}\n")

--- Question 4 ---
Validation Accuracy: 0.64



In [13]:
print(f"--- Question 5 ---")
diffs = {}
for feature in all_features:
    subset_features = [f for f in all_features if f != feature]

    t_dicts = df_train[subset_features].to_dict(orient='records')
    v_dicts = df_val[subset_features].to_dict(orient='records')

    dv_sub = DictVectorizer(sparse=False)
    X_tr_sub = dv_sub.fit_transform(t_dicts)
    X_val_sub = dv_sub.transform(v_dicts)

    m_sub = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    m_sub.fit(X_tr_sub, y_train)

    y_p_sub = m_sub.predict(X_val_sub)
    sub_acc = (y_p_sub == y_val).mean()

    # الفرق بين الدقة الأصلية والدقة بدون الميزة
    diffs[feature] = original_acc - sub_acc

q5_candidates = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
for cand in q5_candidates:
    print(f"Difference when removing '{cand}': {diffs.get(cand, 'N/A')}")
print()

--- Question 5 ---
Difference when removing 'lead_source': 0.0030000000000000027
Difference when removing 'number_of_courses_viewed': 0.0020000000000000018
Difference when removing 'interaction_count': 0.04400000000000004



In [14]:
print(f"--- Question 6 ---")
c_values = [0.000001, 0.00001, 0.0001, 0.001]
for C in c_values:
    m_reg = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    m_reg.fit(X_train, y_train)

    y_p_reg = m_reg.predict(X_val)
    acc_reg = (y_p_reg == y_val).mean()
    print(f"C={C}: Accuracy = {round(acc_reg, 3)}")

--- Question 6 ---
C=1e-06: Accuracy = 0.598
C=1e-05: Accuracy = 0.598
C=0.0001: Accuracy = 0.613
C=0.001: Accuracy = 0.645
